# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token Loaded Successfully!")

Token Loaded Successfully!


In [3]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Connected!")

DuckDB Connected!


In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print(TABLES)

{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [5]:
print("con =", con)
print("TABLES =", TABLES)

con = <duckdb.duckdb.DuckDBPyConnection object at 0x7dd8bb573030>
TABLES = {'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I will prioritize pages that have high search visibility but receive relatively few clicks, followed by pages with low CTR and weaker average search position.

Reason codes:

- HIGH_IMPRESSIONS_LOW_CLICKS: the page receives many impressions but relatively few clicks.
- LOW_CTR: the page has a low click-through rate.
- HIGH_AVG_POSITION: the page has a weaker average search position.

The queue is intended to help the content team decide which pages should receive human review first.

In [6]:


import numpy as np
import pandas as pd


df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE gsc_impressions IS NOT NULL
  AND gsc_impressions > 0
LIMIT 100000
""").df()

print("Rows loaded:", len(df))



df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)


df["reason_code"] = "REVIEW"



df.loc[
    df["ctr"] < 0.03,
    "reason_code"
] = "LOW_CTR"



df.loc[
    df["gsc_avg_position"] > 10,
    "reason_code"
] = "HIGH_AVG_POSITION"



impression_threshold = df["gsc_impressions"].quantile(0.75)

df.loc[
    (df["gsc_impressions"] >= impression_threshold) &
    (df["ctr"] < 0.03),
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CLICKS"



df["priority_score"] = (
    df["gsc_impressions"] * (1 - df["ctr"])
)



queue = df.sort_values(
    "priority_score",
    ascending=False
).head(20)

print("\n===== TOP 20 ACTION QUEUE =====")

print(
    queue[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "reason_code",
            "priority_score"
        ]
    ].to_string(index=False)
)



print("\n===== REASON CODE COUNTS =====")

print(
    df["reason_code"].value_counts()
)

print("\nSection 1 completed successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 100000

===== TOP 20 ACTION QUEUE =====
         content_hash_id  gsc_impressions  gsc_clicks      ctr  gsc_avg_position                 reason_code  priority_score
content_3f9ce33a482237fd              945           0 0.000000          9.108995 HIGH_IMPRESSIONS_LOW_CLICKS           945.0
content_ada72ae2a2b33800              937           3 0.003202          5.597652 HIGH_IMPRESSIONS_LOW_CLICKS           934.0
content_690b092cf66bc2a4              912          14 0.015351          1.641447 HIGH_IMPRESSIONS_LOW_CLICKS           898.0
content_7aa3a33d9d18659e              827           1 0.001209          7.626360 HIGH_IMPRESSIONS_LOW_CLICKS           826.0
content_690b092cf66bc2a4              818          13 0.015892          1.553790 HIGH_IMPRESSIONS_LOW_CLICKS           805.0
content_886a5a08c5ec9c12              787           3 0.003812          7.756036 HIGH_IMPRESSIONS_LOW_CLICKS           784.0
content_690b092cf66bc2a4              755          18 0.023841          

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use and limits

This action queue is intended to help the content team prioritize pages for human review. It uses observed impressions, clicks, CTR, and average position to provide directional decision-support. It should not be treated as proof that a page will improve after optimization, and the recommendations should be reviewed by a person before action.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended users: Content and SEO teams")
print("Purpose: Prioritize pages for human review")
print("Decision type: Directional decision-support")
print("Limit: Recommendations require human review and should not be treated as causal proof.")

print("\nSection 2 completed successfully!")


Intended users: Content and SEO teams
Purpose: Prioritize pages for human review
Decision type: Directional decision-support
Limit: Recommendations require human review and should not be treated as causal proof.

Section 2 completed successfully!


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before taking action, a content team member should review the page, search intent, current title and description, content quality, and the reason code. The queue is decision-support only and should not automatically publish, delete, or rewrite content.

No-go list:
- Do not automatically delete content.
- Do not automatically change important pages.
- Do not automatically publish SEO changes.
- Do not make decisions using the reason code alone.
- Do not treat the priority score as proof of future performance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("===== HUMAN REVIEW CHECK =====")

checks = [
    "Review page content and search intent",
    "Review title and description",
    "Review reason code",
    "Review current search position",
    "Human approval required before changes"
]

for i, check in enumerate(checks, 1):
    print(f"{i}. {check}")

print("\n===== NO-GO LIST =====")

no_go = [
    "Automatic content deletion",
    "Automatic publishing",
    "Automatic major page changes",
    "Using reason code as the only decision",
    "Treating priority score as guaranteed future performance"
]

for item in no_go:
    print("NO:", item)

print("\nSection 3 completed successfully!")


===== HUMAN REVIEW CHECK =====
1. Review page content and search intent
2. Review title and description
3. Review reason code
4. Review current search position
5. Human approval required before changes

===== NO-GO LIST =====
NO: Automatic content deletion
NO: Automatic publishing
NO: Automatic major page changes
NO: Using reason code as the only decision
NO: Treating priority score as guaranteed future performance

Section 3 completed successfully!


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored over time to check whether the signals remain useful.

Retraining or review should be considered if CTR, impressions, clicks, or average position distributions change substantially, if the reason-code mix changes sharply, or if the action queue stops identifying useful pages.

The model or rules should also be reviewed when new data sources or major search changes affect the observed signals.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("===== MONITORING / RETRAIN TRIGGERS =====")

triggers = [
    "Large change in CTR distribution",
    "Large change in impressions distribution",
    "Large change in average-position distribution",
    "Sharp change in reason-code proportions",
    "Action queue becomes less useful to the content team",
    "New data sources or major search changes"
]

for i, trigger in enumerate(triggers, 1):
    print(f"{i}. {trigger}")

print("\nMonitoring status: REVIEW WHEN SIGNALS DRIFT")
print("Section 4 completed successfully!")


===== MONITORING / RETRAIN TRIGGERS =====
1. Large change in CTR distribution
2. Large change in impressions distribution
3. Large change in average-position distribution
4. Sharp change in reason-code proportions
5. Action queue becomes less useful to the content team
6. New data sources or major search changes

Monitoring status: REVIEW WHEN SIGNALS DRIFT
Section 4 completed successfully!


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported as a CSV file so it can be reused in the paper and reviewed separately from the notebook. The export contains the ranked recommendations, observed metrics, reason codes, and priority scores.

In [10]:


import os

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Export the Top-20 queue
output_path = "work/outputs/ml10_action_queue.csv"

queue.to_csv(
    output_path,
    index=False
)

print("===== EXPORT COMPLETE =====")
print("File:", output_path)
print("Rows exported:", len(queue))

# Quick verification
check = pd.read_csv(output_path)

print("\nExported columns:")
print(check.columns.tolist())

print("\nFirst 5 exported rows:")
print(check.head())

print("\nSection 5 completed successfully!")

===== EXPORT COMPLETE =====
File: work/outputs/ml10_action_queue.csv
Rows exported: 20

Exported columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr', 'reason_code', 'priority_score']

First 5 exported rows:
            client_hash_id           content_hash_id  gsc_impressions  \
0  client_73cda7b4e4f265ea  content_3f9ce33a482237fd              945   
1  client_73cda7b4e4f265ea  content_ada72ae2a2b33800              937   
2  client_73cda7b4e4f265ea  content_690b092cf66bc2a4              912   
3  client_73cda7b4e4f265ea  content_7aa3a33d9d18659e              827   
4  client_73cda7b4e4f265ea  content_690b092cf66bc2a4              818   

   gsc_clicks  gsc_avg_position       ctr                  reason_code  \
0           0          9.108995  0.000000  HIGH_IMPRESSIONS_LOW_CLICKS   
1           3          5.597652  0.003202  HIGH_IMPRESSIONS_LOW_CLICKS   
2          14          1.641447  0.015351  HIGH_IMPRESSIONS_LOW_CLICKS   
3   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.